In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

lap_bronze_df = (
    spark.table(bronze_table)
    .filter(F.col("entity_type") == "lap_times")
)

# Infer the nested JSON structure from one lap-response page.
sample_json = (
    lap_bronze_df
    .select("payload_json")
    .first()["payload_json"]
)

lap_schema = (
    spark.range(1)
    .select(
        F.schema_of_json(F.lit(sample_json)).alias("json_schema")
    )
    .first()["json_schema"]
)

parsed_laps_df = lap_bronze_df.withColumn(
    "json_data",
    F.from_json(
        F.col("payload_json"),
        lap_schema
    )
)

In [0]:
race_rows_df = (
    parsed_laps_df
    .select(
        F.explode(
            "json_data.MRData.RaceTable.Races"
        ).alias("race")
    )
)

lap_rows_df = (
    race_rows_df
    .select(
        F.col("race.season").cast("int").alias("season"),
        F.col("race.round").cast("int").alias("round"),
        F.explode("race.Laps").alias("lap")
    )
)

timing_rows_df = (
    lap_rows_df
    .select(
        "season",
        "round",
        F.col("lap.number").cast("int").alias("lap_number"),
        F.explode("lap.Timings").alias("timing")
    )
)

In [0]:
silver_laps_df = (
    timing_rows_df
    .select(
        F.concat_ws(
            "_",
            F.col("season").cast("string"),
            F.lpad(F.col("round").cast("string"), 2, "0")
        ).alias("race_id"),

        "season",
        "round",

        F.col("timing.driverId").alias("driver_id"),
        "lap_number",
        F.col("timing.position").cast("int").alias("position"),
        F.col("timing.time").alias("lap_time_text")
    )
    .withColumn(
        "lap_time_ms",
        (
            F.split("lap_time_text", ":").getItem(0).cast("long") * 60000
            + F.split("lap_time_text", ":").getItem(1).cast("double") * 1000
        ).cast("long")
    )
)

display(
    silver_laps_df
    .orderBy("driver_id", "lap_number")
)

In [0]:
# 1. Duplicate check:
# one driver should have only one record per lap.
duplicate_laps_df = (
    silver_laps_df
    .groupBy("race_id", "driver_id", "lap_number")
    .count()
    .filter("count > 1")
)

print("Duplicate lap keys:", duplicate_laps_df.count())
display(duplicate_laps_df)

In [0]:
# 2. Lap numbers must be positive and reasonable.
invalid_laps_df = silver_laps_df.filter(
    F.col("lap_number").isNull()
    | (F.col("lap_number") <= 0)
    | (F.col("lap_number") > 250)
)

print("Invalid lap numbers:", invalid_laps_df.count())
display(invalid_laps_df)

In [0]:
# 3. Lap times must be present and positive.
invalid_times_df = silver_laps_df.filter(
    F.col("lap_time_text").isNull()
    | F.col("lap_time_ms").isNull()
    | (F.col("lap_time_ms") <= 0)
)

print("Invalid lap times:", invalid_times_df.count())
display(invalid_times_df)

In [0]:
catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

silver_lap_table = f"{catalog}.f1_silver.lap_times"

silver_laps_to_save_df = silver_laps_df.withColumn(
    "loaded_at",
    F.current_timestamp()
)

(
    silver_laps_to_save_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_lap_table)
)

print(f"Saved Silver lap table: {silver_lap_table}")

In [0]:
saved_laps_df = spark.table(silver_lap_table)

print("Lap rows:", saved_laps_df.count())

display(
    saved_laps_df
    .select(
        "race_id",
        "driver_id",
        "lap_number",
        "lap_time_text",
        "lap_time_ms"
    )
    .orderBy("driver_id", "lap_number")
)